# C4 — Beta sweep: does the Laplace-NLL reweighting help calibration, for every architecture?

`C3_beta_sweep.ipynb` ran this comparison for `unet_nll` alone and found that the
project default **`beta = 0.5` was the best-calibrated point** (`z_std` 0.909, the
closest to 1.0; `beta = 0.75` badly over-dispersed at 0.607). This notebook repeats
it for **all four base Laplacian NLL architectures** — `unet_nll`, `resunet_nll`,
`attention_unet_nll`, `efficientnet_unet_nll` — so the question "should any
architecture train at a beta other than 0.5?" is answered per-architecture rather
than extrapolated from the pilot.

**Checkpoints.** `beta = 0.5` is each architecture's standing `models/nll/<arch>/`
run (`021`/`022`); the other betas come from `024` (`unet_nll`) and `025` (the other
three), under `models/beta_sweep/beta_<value>/<arch>/`. A missing checkpoint is
skipped with a message, not an error — so this notebook is readable before `025` has
finished.

**Same machinery as `C0`/`C1`/`C3`**: fidelity (§5), detection AUROC (§6), stroke
coherence (§7), sigma calibration (§8), a coarse verdict tally (§9). Every column is
the same Laplace distribution, so — unlike `C1`'s Gaussian-vs-Laplace case —
`evaluate_calibration`'s `nll` field *is* directly comparable across betas (it is the
plain unweighted Laplace NLL, not the beta-reweighted training loss).

**Read §5 and §8 before §9.** §6's AUROC rests on three masks and, across `C0`+`C1`,
correlates with *worse* reconstruction (Spearman `rho = +0.665`, `p = 0.005`) — a
confound not ruled out for beta, so a beta that wins detection and loses fidelity is
not obviously better.

Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports and a GPU sanity check. Same building blocks as `C0`/`C1`/`C3`; `scripts.trainer` (the deterministic loader) is not needed — every column is an NLL head.

In [ ]:
import gc

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

from scripts.calibration import (
    evaluate_calibration,
    laplace_sigma_from_scale,
    learned_zscore,
    structural_zscore,
)
from scripts.config import settings
from scripts.dataset import pad_to_multiple
from scripts.delta_analysis import analyze_delta
from scripts.detection import evaluate_detection
from scripts.stroke_stats import stroke_coherence
from scripts.trainer_nll import load_model_nll
from scripts.visualization_nll import DEFAULT_Z_SCALE, plot_signal_comparison

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")

## 1. The `data/test/` images

Every `(rgb, ir)` pair under `data/test/`, discovered generically — same as
`040`/`C0`/`C1`/`C3`. The three with a hand-drawn mask (`GT01`/`GT02`/`GT03`)
additionally carry a detection ground truth; fidelity/coherence/calibration need no
mask and run on all of them.

In [ ]:
TEST_RGB_DIR = project_root / "data" / "test" / "rgb"
TEST_IR_DIR = project_root / "data" / "test" / "ir"
ANNOTATIONS_DIR = project_root / "data" / "test" / "annotations"

rgb_paths = sorted(TEST_RGB_DIR.glob("*.jpg")) + sorted(TEST_RGB_DIR.glob("*.png"))
image_pairs = [
    (p, TEST_IR_DIR / p.name)
    for p in sorted(rgb_paths)
    if (TEST_IR_DIR / p.name).exists()
]
gt_stems = {
    p.name.removesuffix("_Map.png") for p in sorted(ANNOTATIONS_DIR.glob("*_Map.png"))
}
gt_stems &= {p.stem for p, _ in image_pairs}

print(f"data/test/ images: {len(image_pairs)} | {[p.stem for p, _ in image_pairs]}")
print(f"with a ground-truth mask: {sorted(gt_stems)}")

if not image_pairs:
    raise RuntimeError("No (rgb, ir) pairs found under data/test/.")

## 2. The architecture x beta grid

`ARCHS` is the four base Laplacian NLL heads (`efficientnet_unet_nll_ft` is left out —
closed as a negative result, `fixing.md` §0). `BETA_RUNS` maps each beta to its
checkpoint root: `beta = 0.5` is `models/nll/` (the project default, from
`021`/`022`), the rest are `024`/`025`'s per-beta directories. Every
`(arch, beta)` with no checkpoint is reported and skipped.

In [ ]:
ARCHS = ["unet_nll", "resunet_nll", "attention_unet_nll", "efficientnet_unet_nll"]
LOSS_NAME = "laplace_nll"
REFERENCE_BETA = settings.NLL_BETA  # 0.5

BETA_RUNS = {
    0.0: settings.MODELS_DIR / "beta_sweep" / "beta_0.00",
    0.25: settings.MODELS_DIR / "beta_sweep" / "beta_0.25",
    REFERENCE_BETA: settings.MODELS_DIR / "nll",
    0.75: settings.MODELS_DIR / "beta_sweep" / "beta_0.75",
}

SIGNAL_KINDS = ("raw delta", "structural delta", "confidence")
NLL_SIGNAL_KINDS = ("|z| (raw/sigma)", "structural z")

grid = [
    (arch, beta)
    for arch in ARCHS
    for beta in BETA_RUNS
]
present = [
    (arch, beta)
    for arch, beta in grid
    if (BETA_RUNS[beta] / arch / "best_model.keras").exists()
]
missing = [pair for pair in grid if pair not in present]

print(f"Architectures: {ARCHS}")
print(f"Betas:         {sorted(BETA_RUNS)}")
print(f"Checkpoints present: {len(present)}/{len(grid)}")
if missing:
    print(f"Missing (will be skipped): {missing}")

## 3. Signals

Identical construction to `040`/`041`/`C3` — **raw delta**, **structural delta**,
**confidence**, plus **`|z|`** and **structural z**, all read off the same
`sigma = laplace_sigma_from_scale(exp(log_b))` conversion (every column is Laplace,
so no per-column dispatch). `pad_to_multiple(..., multiple=settings.PATCH_MULTIPLE)`
is used for every architecture, `efficientnet_unet_nll` included — its decoder's
`_ResizeToMatch` layers absorb the residual size mismatch, same as `041`/`052`.

In [ ]:
MASK_THRESHOLD = 127  # midpoint threshold for the mask's anti-aliased edges


def load_pair(rgb_path: Path, ir_path: Path) -> tuple[np.ndarray, np.ndarray]:
    """Load an ``(rgb, ir)`` pair as float arrays in ``[0, 1]``."""
    rgb = np.array(Image.open(rgb_path).convert("RGB")).astype(np.float32) / 255.0
    ir = np.array(Image.open(ir_path).convert("L")).astype(np.float32) / 255.0
    return rgb, ir


def load_mask(stem: str) -> np.ndarray:
    """Load a hand-drawn ground-truth mask as a boolean array."""
    mask_path = ANNOTATIONS_DIR / f"{stem}_Map.png"
    return np.array(Image.open(mask_path).convert("L")) > MASK_THRESHOLD


def fidelity(ir: np.ndarray, mu: np.ndarray) -> dict[str, float]:
    """MAE/SSIM/PSNR of a prediction against the real IR, as in 030."""
    real = tf.constant(ir[..., np.newaxis])
    pred = tf.constant(mu[..., np.newaxis])
    return {
        "mae": float(np.mean(np.abs(ir - mu))),
        "ssim": float(tf.image.ssim(real, pred, max_val=1.0)),
        "psnr": float(tf.image.psnr(real, pred, max_val=1.0)),
    }


def predict_signals(
    model: tf.keras.Model, rgb: np.ndarray, ir: np.ndarray
) -> tuple[dict[str, float], dict[str, np.ndarray], np.ndarray, np.ndarray]:
    """Predict one image and build every signal this architecture supports."""
    padded, _ = pad_to_multiple(tf.constant(rgb), multiple=settings.PATCH_MULTIPLE)
    h, w = ir.shape
    pred = model.predict(padded[tf.newaxis, ...], verbose=0)[0, :h, :w, :]

    mu = pred[..., 0]
    sigma = laplace_sigma_from_scale(np.exp(pred[..., 1]))
    result = analyze_delta(ir, mu)
    signals = {
        "raw delta": result.raw_delta,
        "structural delta": result.structural_delta,
        "confidence": result.confidence_map,
        "|z| (raw/sigma)": np.abs(learned_zscore(ir, mu, sigma)),
        "structural z": structural_zscore(result.structural_delta, sigma),
    }
    return fidelity(ir, mu), signals, sigma, mu

## 4. The sweep

One `(architecture, beta)` at a time: loaded, scored over every `data/test/` image,
released before the next. Results are keyed by `(arch, beta)` so §5-§9 can slice them
per architecture.

`PLOT_IMAGES`/`PLOT_SIGNALS` control only the figures; the tables always cover every
image and signal.

In [ ]:
PLOT_IMAGES = set()   # -> set(gt_stems) to plot the ground-truth images
PLOT_SIGNALS = set()  # -> {"raw delta", "structural delta"} to plot those

fidelity_rows: dict[tuple[str, float], dict[str, list[float]]] = {}
coherence_rows: dict[tuple[str, float, str], list[float]] = {}
auroc_rows: dict[tuple[str, float, str], dict[str, float]] = {}
ap_rows: dict[tuple[str, float, str], dict[str, float]] = {}
calibration_rows: dict[tuple[str, float], dict[str, list[float]]] = {}

compared: list[tuple[str, float]] = []

for arch in ARCHS:
    for beta, model_dir in BETA_RUNS.items():
        try:
            model = load_model_nll(
                arch, model_dir=model_dir, loss_name=LOSS_NAME, beta=beta
            )
        except FileNotFoundError as exc:
            print(f"[skip] {arch} beta={beta}: {exc}")
            continue

        compared.append((arch, beta))
        tag = " (reference)" if beta == REFERENCE_BETA else ""
        print(f"\n=== {arch} — beta={beta}{tag} ===")

        for rgb_path, ir_path in image_pairs:
            stem = rgb_path.stem
            rgb, ir = load_pair(rgb_path, ir_path)
            mask = load_mask(stem) if stem in gt_stems else None

            scores, signals, sigma, mu = predict_signals(model, rgb, ir)

            for metric, value in scores.items():
                fidelity_rows.setdefault((arch, beta), {}).setdefault(metric, []).append(value)

            for kind, signal in signals.items():
                coherence_rows.setdefault((arch, beta, kind), []).append(
                    stroke_coherence(signal).coherence
                )
                if mask is not None:
                    detection = evaluate_detection(signal, mask)
                    auroc_rows.setdefault((arch, beta, kind), {})[stem] = detection.auroc
                    ap_rows.setdefault((arch, beta, kind), {})[stem] = detection.average_precision

            summary = evaluate_calibration(ir, mu, sigma, distribution="laplace").summary()
            for metric, value in summary.items():
                calibration_rows.setdefault((arch, beta), {}).setdefault(metric, []).append(value)

            if stem in PLOT_IMAGES:
                for kind in PLOT_SIGNALS:
                    fig = plot_signal_comparison(
                        ir, {f"{arch} b={beta}": signals[kind]},
                        title=f"{arch} — {kind} — {stem} — beta={beta}",
                        vrange=(0.0, 1.0),
                    )
                    plt.show()
                    plt.close(fig)

            print(f"  {stem}: scored")
            del signals, rgb, ir, mask

        del model
        gc.collect()
        tf.keras.backend.clear_session()

print(f"\nCompared: {compared}")
if not compared:
    raise RuntimeError("No (arch, beta) had a checkpoint to load — nothing to compare.")

betas_seen = sorted({b for _, b in compared})

## 5. Reconstruction fidelity, per architecture

`mae`/`ssim`/`psnr` of `mu` against the real IR, averaged over all `data/test/`
images — the axis `030` reports. `delta` is against that architecture's own
`beta = 0.5` column, oriented so positive always means "improvement". These metrics
do not carry the beta weight, so they are the first honest read on which beta
trained best on fidelity alone.

In [ ]:
LOWER_IS_BETTER = {"mae": True, "ssim": False, "psnr": False}


def gain(metric: str, ref: float, other: float) -> float:
    return ref - other if LOWER_IS_BETTER[metric] else other - ref


def arch_betas(arch: str) -> list[float]:
    return [b for a, b in compared if a == arch]


for arch in ARCHS:
    betas = sorted(arch_betas(arch))
    if not betas:
        print(f"\n{arch}: no checkpoints\n")
        continue
    has_ref = REFERENCE_BETA in betas

    print(f"\n{arch}")
    header = "  metric".ljust(12) + "".join(f"beta={b}".rjust(20) for b in betas)
    print(header)
    print("  " + "-" * (len(header) - 2))

    for metric in LOWER_IS_BETTER:
        row = f"  {metric}".ljust(12)
        ref_value = (
            float(np.mean(fidelity_rows[(arch, REFERENCE_BETA)][metric])) if has_ref else None
        )
        for beta in betas:
            value = float(np.mean(fidelity_rows[(arch, beta)][metric]))
            if has_ref and beta != REFERENCE_BETA:
                row += f"{value:.4f} ({gain(metric, ref_value, value):+.4f})".rjust(20)
            else:
                row += f"{value:.4f}".rjust(20)
        print(row)

print(f"\n(mean over {len(image_pairs)} data/test/ images; delta vs. that arch's beta={REFERENCE_BETA})")

## 6. Detection AUROC, per architecture

AUROC of each signal against the hand-drawn masks, averaged over `GT01`/`GT02`/`GT03`
only. `0.5` is chance. Read with §5/§8 — `C0`+`C1` found AUROC correlates with
*worse* reconstruction, a confound not ruled out for beta. `n = 3`, so the mean
hides a lot.

In [ ]:
def detection_table(rows: dict, arch: str, label: str) -> None:
    betas = sorted(arch_betas(arch))
    kinds = [k for k in (*SIGNAL_KINDS, *NLL_SIGNAL_KINDS)
             if any((arch, b, k) in rows for b in betas)]
    name_w = 24
    header = "  signal".ljust(name_w) + "".join(f"beta={b}".rjust(12) for b in betas)
    print(header)
    print("  " + "-" * (len(header) - 2))
    for kind in kinds:
        if not all((arch, b, kind) in rows for b in betas):
            continue
        row = f"  {kind}".ljust(name_w)
        for beta in betas:
            row += f"{float(np.mean(list(rows[(arch, beta, kind)].values()))):.4f}".rjust(12)
        print(row)
    print(f"  ({label}, mean over {sorted(gt_stems)})")


if auroc_rows:
    for arch in ARCHS:
        if not arch_betas(arch):
            continue
        print(f"\n{arch} — AUROC")
        detection_table(auroc_rows, arch, "AUROC")
        print(f"\n{arch} — average precision")
        detection_table(ap_rows, arch, "average precision")
else:
    print("No ground-truth mask found under data/test/annotations/ — section skipped.")

### Per-image AUROC breakdown

The `n = 3` mean can be carried entirely by one favourable image. Shown per
architecture, `structural z` and `|z|` only (the sigma-dependent signals — the ones
beta actually moves).

In [ ]:
if auroc_rows:
    stems = sorted(gt_stems)
    for arch in ARCHS:
        betas = sorted(arch_betas(arch))
        if not betas:
            continue
        print(f"\n{arch}")
        name_w = 20
        header = "  signal".ljust(name_w) + "".join(
            f"{stem} b={b}".rjust(13) for stem in stems for b in betas
        )
        print(header)
        print("  " + "-" * (len(header) - 2))
        for kind in NLL_SIGNAL_KINDS:
            if not all((arch, b, kind) in auroc_rows for b in betas):
                continue
            row = f"  {kind}".ljust(name_w)
            for stem in stems:
                for beta in betas:
                    row += f"{auroc_rows[(arch, beta, kind)][stem]:.3f}".rjust(13)
            print(row)
else:
    print("No ground-truth mask found — section skipped.")

## 7. Stroke coherence, per architecture

How much of each signal is oriented, line-like structure rather than isotropic noise
(`scripts.stroke_stats`) — no mask needed, so it runs on all ten `data/test/` images
and is a structurally independent check on the AUROC ranking.

In [ ]:
for arch in ARCHS:
    betas = sorted(arch_betas(arch))
    if not betas:
        continue
    print(f"\n{arch}")
    name_w = 24
    header = "  signal".ljust(name_w) + "".join(f"beta={b}".rjust(18) for b in betas)
    print(header)
    print("  " + "-" * (len(header) - 2))
    for kind in (*SIGNAL_KINDS, *NLL_SIGNAL_KINDS):
        if not all((arch, b, kind) in coherence_rows for b in betas):
            continue
        row = f"  {kind}".ljust(name_w)
        for beta in betas:
            values = np.array(coherence_rows[(arch, beta, kind)])
            row += f"{values.mean():.3f}±{values.std():.3f}".rjust(18)
        print(row)

print(f"\n(mean ± std over {len(image_pairs)} data/test/ images)")

## 8. Sigma calibration, per architecture — the axis this sweep is about

No training-time metric measures sigma behaviour, so this is where the beta question
gets settled. `nll` **is** comparable across columns here (same Laplace density) but
is not the target — beta trades `nll` for weighting, so a lower `nll` at low beta is
expected and not by itself an improvement. Judge on `z_std` (toward `1.0`), `ence`
(down), `error_sigma_spearman` (up), coverage (near nominal: 1s = 0.6827,
2s = 0.9545).

In [ ]:
CALIB_KEYS = [
    "nll", "ence", "z_std", "coverage_1s", "coverage_2s",
    "error_sigma_spearman", "sharpness", "dispersion",
]

for arch in ARCHS:
    betas = sorted(arch_betas(arch))
    if not betas:
        continue
    print(f"\n{arch}")
    name_w = 24
    header = "  metric".ljust(name_w) + "".join(f"beta={b}".rjust(14) for b in betas)
    print(header)
    print("  " + "-" * (len(header) - 2))
    for metric in CALIB_KEYS:
        row = f"  {metric}".ljust(name_w)
        for beta in betas:
            row += f"{float(np.mean(calibration_rows[(arch, beta)][metric])):.4f}".rjust(14)
        print(row)

print("\nnominal coverage: 1s = 0.6827, 2s = 0.9545; calibrated z_std = 1.0")
print(f"reference column: beta={REFERENCE_BETA} (project-wide default, from 021/022)")

## 9. Verdict — per architecture, which beta moved anything

Per architecture: how many measured quantities improved **relative to that
architecture's own `beta = 0.5`**. Deliberately coarse — it collapses fidelity,
detection and coherence (which need not agree) and §6 carries the confound above.
Read §5 and §8 first.

The final line is the one that matters for k-fold: **does any architecture have a
beta that clearly beats 0.5 on `z_std`** (the primary calibration target)? If not,
keep `beta = 0.5` fixed everywhere — a per-architecture beta tuned on this test set
would compromise a k-fold reported on the same data.

In [ ]:
def tally(gains: list[float]) -> str:
    if not gains:
        return "-"
    return f"{sum(g > 0 for g in gains)}/{len(gains)}"


ALL_KINDS = (*SIGNAL_KINDS, *NLL_SIGNAL_KINDS)
CALIB_HIGHER_IS_BETTER = {"z_std": False, "ence": False, "error_sigma_spearman": True}


def calib_gain(arch, beta, metric, higher_is_better):
    ref = float(np.mean(calibration_rows[(arch, REFERENCE_BETA)][metric]))
    val = float(np.mean(calibration_rows[(arch, beta)][metric]))
    return (val - ref) if higher_is_better else (ref - val)


for arch in ARCHS:
    betas = sorted(arch_betas(arch))
    if REFERENCE_BETA not in betas:
        print(f"\n{arch}: no beta=0.5 reference — skipped")
        continue

    print(f"\n{arch}")
    print("  beta".ljust(12) + "fidelity".rjust(12) + "detection".rjust(12)
          + "coherence".rjust(12) + "calibration".rjust(14))
    print("  " + "-" * 60)
    for beta in betas:
        if beta == REFERENCE_BETA:
            print(f"  {beta}".ljust(12) + "(reference)".rjust(50))
            continue
        fid = [
            gain(m, float(np.mean(fidelity_rows[(arch, REFERENCE_BETA)][m])),
                 float(np.mean(fidelity_rows[(arch, beta)][m])))
            for m in LOWER_IS_BETTER
        ]
        det = [
            float(np.mean(list(auroc_rows[(arch, beta, k)].values())))
            - float(np.mean(list(auroc_rows[(arch, REFERENCE_BETA, k)].values())))
            for k in ALL_KINDS
            if (arch, beta, k) in auroc_rows and (arch, REFERENCE_BETA, k) in auroc_rows
        ]
        coh = [
            float(np.mean(coherence_rows[(arch, beta, k)]))
            - float(np.mean(coherence_rows[(arch, REFERENCE_BETA, k)]))
            for k in ALL_KINDS
            if (arch, beta, k) in coherence_rows
        ]
        cal = [calib_gain(arch, beta, m, hib) for m, hib in CALIB_HIGHER_IS_BETTER.items()]
        print(f"  {beta}".ljust(12) + tally(fid).rjust(12) + tally(det).rjust(12)
              + tally(coh).rjust(12) + tally(cal).rjust(14))

print("\n\n=== best beta per architecture on z_std (target 1.0) ===")
for arch in ARCHS:
    betas = sorted(arch_betas(arch))
    if not betas:
        continue
    z = {b: float(np.mean(calibration_rows[(arch, b)]["z_std"])) for b in betas}
    best = min(z, key=lambda b: abs(z[b] - 1.0))
    ref_note = "" if best == REFERENCE_BETA else "  <-- differs from default"
    cells = "  ".join(f"b{b}={z[b]:.3f}" for b in betas)
    print(f"  {arch:<24} {cells}   -> best beta={best}{ref_note}")

## 10. Conclusion

_Fill in after running._ Template:

- **Per-architecture best beta on `z_std`**: …
- **Does `resunet_nll` (the k-fold candidate) want a beta other than 0.5?** …
- **Any architecture where a non-0.5 beta wins on `z_std` *and* does not lose
  fidelity (§5)?** …
- **Decision for Round 4**: keep `beta = 0.5` fixed across all NLL architectures /
  use `beta = X` for architecture `Y` (with the justification that it is not test-set
  tuning because …).